### IRFL -gMLP/GRU testing

In [1]:
import sys
import os

sys.path.append(os.getcwd())
# append the root directory to the sys.path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

import torch
from src.utils.irfl_dataset import IRFLDataset, make_dataset
from src.models.jointopt import JointOpt
from src.models.repercent import DisenLoss
from src.models.third_party.g_mlp_repo.g_mlp import gMLP
from training.train_repercent import train_loop, make_dataloaders
from training.train_jointopt_2m import make_model_jointopt
import yaml

datasets_path = "../data/irfl/datasets/"

### Load the dataset

In [7]:
# Load and test data
M = 3 # nuber of modalities
total_train_data = torch.load(os.path.join(datasets_path, 'IRFL_train_tensors_2.pt'), map_location="cpu")
total_test_data = torch.load(os.path.join(datasets_path, 'IRFL_test_tensors_2.pt'), map_location="cpu")

total_train_data_aug = torch.load(os.path.join(datasets_path, 'IRFL_train_tensors_aug_2.pt'), map_location="cpu")
total_test_data_aug = torch.load(os.path.join(datasets_path, 'IRFL_test_tensors_aug_2.pt'), map_location="cpu")

train_dataset, _ = make_dataset(total_data= total_train_data | total_train_data_aug, num_modalities= M, data_type='train', include_original=True)
test_dataset, _ = make_dataset(total_data= total_test_data | total_test_data_aug, num_modalities= M, data_type='test', include_original=True)

### Read configs & make model

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"


with open(f'../configs/data/irfl_data_{M}m.yaml', 'r') as f:
    data_config = yaml.safe_load(f)
with open(f'../configs/model/gmlp_irfl_{M}m.yaml', 'r') as f:
    model_config = yaml.safe_load(f)
with open(f'../configs/training/train_irfl_{M}m.yaml', 'r') as f:
    training_config = yaml.safe_load(f)

model = make_model_jointopt(model_config).to(device)


disen_loss = DisenLoss(alpha=training_config["disen_loss"]["alpha"],
                    lmd=training_config["disen_loss"]["lmd"],
                    lmd_end_value=training_config["disen_loss"]["lmd_end_value"],
                    M=data_config["create_data"]["M"],
                    recon= training_config["disen_loss"]["recon"])
print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")

Projection heads defined for gMLP encoders: True. Projection head dimensions: [[768, 256], [768, 256], [512, 256], [512, 256], [512, 256], [512, 256]]
Projection heads defined for gMLP encoders: True. Projection head dimensions: [[768, 256], [768, 256], [512, 256], [512, 256], [512, 256], [512, 256]]
Built 6 shared encoders and 6 unique encoders for JointOpt model.
Model initialized with latent dimension: 256 and sequence dimension: 49
Model initialized with 13031408 parameters


In [1]:
%pip install fvcore

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61443 sha256=42bba51411ee5f3f89168cf17703942c4b242ede3b46860a8f6d53ca157c4ad6
  Stored in directory: /tmp/pip-ephem-wheel-cache-cpqk9g6y/wheels/01/c0/af/77c1cf53a1be9e42a52b48e5af2169d40ec2e89f7362489dd0
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31597 sha256=be8b41fe44648501748492d5a313df7f877a1c5782517a0922edd3ff2133f49e
  Stored in directory: /tmp/pip-ephem-wheel-cache-cpqk9g6y/wheels/9a/a3/b6/ac0fcd1b4ed5cfeb3db92e6a0e476cfd48ed0df92b91080c1d
Successfully 

In [9]:
from fvcore.nn import FlopCountAnalysis

def calc_flops(model, inputs):
    """
    Args:
        model: torch.nn.Module
        inputs: tuple of inputs as passed to model(*inputs)
    Returns:
        FLOPs (int)
    """
    model.eval()
    flops = FlopCountAnalysis(model, inputs)
    return flops.total()

# Example usage:
random_images = torch.randn(32, 49, 768)  # Example image input
random_texts = torch.randn(32, 77, 512)  # Example text input
random_defs = torch.randn(32, 77, 512)  # Example definitions input
text_mask = torch.ones(32, 77)  # Example text mask
random_defs_mask = torch.ones(32, 77)  # Example definitions mask
X = [random_images.to(device), random_texts.to(device), random_defs.to(device)]
X_cross_masks = [None, text_mask.bool().to(device), random_defs_mask.bool().to(device)] 
inputs = (X, X_cross_masks)  # Example input shapes for the three modalities
flops = calc_flops(model, inputs)
print(f"Total FLOPs for a forward pass: {flops}")

/mnt/lts4-dislearn/scratch/home/rizou/lts4-FaP/src/DisentangledSSL/models.py:137: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  con1 = torch.tensor((self.__m - 1) / 2, dtype=torch.float64)
/mnt/lts4-dislearn/scratch/home/rizou/lts4-FaP/src/DisentangledSSL/models.py:138: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  con2 = torch.tensor((self.__m - 1) / 2, dtype=torch.float64)
Unsupported operator aten::gelu encountered 24 time(s)
Unsupported operator aten::mul encountered 240 time(s)
Unsupported operator aten::add encountered 102 time(s)
Unsupported operator aten::sum encountered 30 time(s)
Unsupported operator aten::div encountered 105 time(s)
Unsupported operator aten::linalg_vector_nor

Total FLOPs for a forward pass: 24109711360


In [11]:
epochs = training_config["training"]["n_epochs"]
BATCH_SIZE = training_config["training"]["batch_size"]
learning_rate = training_config["optimizer"]["lr"]
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_loader, test_loader = make_dataloaders(train_dataset, test_dataset, batch_size=BATCH_SIZE)


for _iter in range(epochs):
    # Initialize epoch loss trackers
    epoch_loss = 0.0
    epoch_ortho_loss = 0.0
    epoch_unique_loss = 0.0
    epoch_shared_loss = 0.0
    
    model.train()
    print(f"----- Epoch: {_iter + 1} / {epochs} -----")
    # Training phase
    for batch_idx, out in enumerate(train_loader):

        x = out['x']
        x_aug = out['x_aug']
        x_orig = out['orig']

        temp_b = x['images'].shape[0]
        images, texts, text_mask = x["images"], x["texts"], x["pad_masks"]
        images_aug, texts_aug, text_mask_aug = x_aug["images"], x_aug["texts"], x_aug["pad_masks"]
        
        X = [images.to(device), texts.to(device)]
        X_cross_masks = [None, text_mask.bool().to(device)] 
        
        X_aug = [images_aug.to(device), texts_aug.to(device)]
        X_aug_cross_masks = [None, text_mask_aug.bool().to(device)]
    
        # Forward pass through RePercENT
        outputs = model(X, mask = X_cross_masks)
        outputs_aug = model(X_aug, mask = X_aug_cross_masks)
        
        # Compute disentanglement loss
        loss, logs = disen_loss(outputs, outputs_aug)
        
        # Backward pass for RePercENT
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # # Track losses
        epoch_loss += loss.item()
        epoch_ortho_loss += logs['ortho']
        epoch_unique_loss += logs['unique']
        epoch_shared_loss += logs['shared']
        
    # Epoch statistics
    avg_epoch_loss = epoch_loss / len(train_loader)
    avg_ortho_loss = epoch_ortho_loss / len(train_loader)
    avg_unique_loss = epoch_unique_loss / len(train_loader)
    avg_shared_loss = epoch_shared_loss / len(train_loader)

    print(f"Epoch {_iter + 1} - Loss: {avg_epoch_loss:.4f}, Ortho Loss: {avg_ortho_loss:.4f}, Unique Loss: {avg_unique_loss:.4f}, Shared Loss: {avg_shared_loss:.4f}")

----- Epoch: 1 / 35 -----
Epoch 1 - Loss: 4.4041, Ortho Loss: 21.2360, Unique Loss: 0.1918, Shared Loss: 3.2392
----- Epoch: 2 / 35 -----
Epoch 2 - Loss: 3.4458, Ortho Loss: 14.5302, Unique Loss: 0.0671, Shared Loss: 2.5567
----- Epoch: 3 / 35 -----
Epoch 3 - Loss: 2.7732, Ortho Loss: 14.0023, Unique Loss: 0.0423, Shared Loss: 2.0588
----- Epoch: 4 / 35 -----
Epoch 4 - Loss: 2.3759, Ortho Loss: 13.5572, Unique Loss: 0.0392, Shared Loss: 1.7619
----- Epoch: 5 / 35 -----
Epoch 5 - Loss: 2.0243, Ortho Loss: 13.3493, Unique Loss: 0.0406, Shared Loss: 1.4980
----- Epoch: 6 / 35 -----
Epoch 6 - Loss: 1.7412, Ortho Loss: 13.0118, Unique Loss: 0.0353, Shared Loss: 1.2874
----- Epoch: 7 / 35 -----
Epoch 7 - Loss: 1.5374, Ortho Loss: 12.8278, Unique Loss: 0.0344, Shared Loss: 1.1348
----- Epoch: 8 / 35 -----
Epoch 8 - Loss: 1.3545, Ortho Loss: 12.7114, Unique Loss: 0.0322, Shared Loss: 0.9983
----- Epoch: 9 / 35 -----
Epoch 9 - Loss: 1.2022, Ortho Loss: 12.6014, Unique Loss: 0.0317, Shared Loss:

In [50]:
# Define test loop
# Initialize epoch loss trackers
from einops import rearrange
import torch.nn.functional as F
from collections import defaultdict


overall_correct = 0
overall_total = 0

# track per type
type_correct = defaultdict(int)
type_total   = defaultdict(int)


model.eval()

# test_loader = torch.utils.data.DataLoader(test_dataset, batch_size= 5, shuffle=False, num_workers=2)
# Testing phase
with torch.no_grad():
    for batch_idx, out in enumerate(test_loader):

        x = out['x']
        x_aug = out['x_aug']
        x_orig = out['orig']
        
        temp_b = x['images'].shape[0]
        images, texts, text_mask = x["images"], x["texts"], x["pad_masks"]
        # defs, defs_mask = x["definitions"], x["definitions_mask"]
        
        X = [images.to(device), texts.to(device)]
        X_cross_masks = [None, text_mask.to(device)]


        distractors = out['distractors'].to(device)

        B, N, S, D = distractors.shape
        distr_flat = rearrange(distractors, 'b n s d -> (b n) s d')
        out_distractors_flat = model.encode_modality(model.sharedEncoders[f"S_12"], \
                                model.sharedProjh[f"S_12"],distr_flat, None)
        

        out_distractors = rearrange(out_distractors_flat, '(b n) ... -> b n ...', b=B, n=N)

        
        # Forward pass through RePercENT for the main inputs
        outputs = model(X, mask = X_cross_masks)
        
        # calculate the shared component similarity between the answer and distractors
        # shared_text = outputs['S_view'][:, 1, 0]  # [B, D]
        # shared_text = shared_text / shared_text.norm(dim=-1, keepdim=True)
        # shared_image_answers = outputs['S_view'][:, 0, 1]  # [B, D]
        # shared_image_answers = shared_image_answers / shared_image_answers.norm(dim=-1, keepdim=True)
        # shared_image_distractors = out_distractors  # [B, num_distractors, D]
        # shared_image_distractors = shared_image_distractors / shared_image_distractors.norm(dim=-1, keepdim=True)
        
        # for i in range(B):
        #     answer_sim = (shared_text[i:i+1] @ shared_image_answers[i:i+1].T).item()
        #     distractor_sims = (shared_text[i:i+1] @ shared_image_distractors[i].T).squeeze(0)  # [num_distractors]
            
        #     max_distractor_sim = torch.max(distractor_sims).item()
        #     if answer_sim > max_distractor_sim:
        #         accuracy += 1.0
        #     cnt += 1
        # shared_text: [B, D]
        shared_text = outputs['S_view'][:, 1, 0]
        shared_text = F.normalize(shared_text, dim=-1)

        # shared_image_answers: [B, D]
        shared_image_answers = outputs['S_view'][:, 0, 1]
        shared_image_answers = F.normalize(shared_image_answers, dim=-1)

        # shared_image_distractors: [B, K, D]
        shared_image_distractors = out_distractors
        shared_image_distractors = F.normalize(shared_image_distractors, dim=-1)

        # answer_sim: [B]
        answer_sim = (shared_text * shared_image_answers).sum(dim=-1)

        # distractor_sims: [B, K]
        # einsum computes dot(shared_text[b], shared_image_distractors[b, k]) for all b,k
        distractor_sims = torch.einsum('bd,bkd->bk', shared_text, shared_image_distractors)

        # max_distractor_sim: [B]
        max_distractor_sim = distractor_sims.max(dim=1).values
        
        # correct: [B] bool
        correct = answer_sim >= max_distractor_sim
        
        # ---- overall ----
        overall_correct += correct.sum().item()
        overall_total += correct.numel()

        # ---- by figurative type ----
        # out["figurative_type"] is usually a list of strings length B
        ftypes = out["figurative_type"]

        for t, c in zip(ftypes, correct.tolist()):
            # normalize spelling just in case: "metaphore" vs "metaphor"
            t = str(t).lower().strip()
            
            type_total[t] += 1
            type_correct[t] += int(c)
            
# final metrics
overall_acc = overall_correct / max(1, overall_total)
print(f"Overall Test Accuracy: {overall_acc*100:.2f}% ({overall_correct}/{overall_total})")


Overall Test Accuracy: 40.49% (328/810)


In [51]:
for t in ["idiom", "metaphor", "simile"]:
    tot = type_total.get(t, 0)
    cor = type_correct.get(t, 0)
    acc = (cor / tot) if tot > 0 else float("nan")
    print(f"{t.capitalize():>9} Accuracy: {acc*100:.2f}% ({cor}/{tot})")

    Idiom Accuracy: 34.50% (69/200)
 Metaphor Accuracy: 29.13% (97/333)
   Simile Accuracy: 58.48% (162/277)
